<a href="https://colab.research.google.com/github/chenharry66/adversial-audio-experiments/blob/main/genshinv_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datasets import load_dataset


ds = load_dataset("simon3000/genshin-voice", split="train", streaming=True)

def select_rows(transcription, speaker, type, language):
    if not transcription or not speaker or type != 'Dialog' or language != 'English(US)' :
        return False

    return True

ds = ds.filter(select_rows, input_columns=['transcription', 'speaker', 'type', 'language'])

ds = ds.take(5000)

README.md:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

In [3]:
from datasets import Dataset

def dataset_generator():
    for item in ds:
        yield item

ds = Dataset.from_generator(dataset_generator)

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
ds.unique('language')

['English(US)']

In [5]:
ds[0]['audio']

{'bytes': b'RIFF\xbe\x05\t\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80\xbb\x00\x00\x00w\x01\x00\x02\x00\x10\x00data\x9a\x05\t\x00\x00\x00\x00\x00\x01\x00\x00\x00\x02\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\xfe\xff\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\xff\xff\x00\x00\xfe\xff\xfe\xff\x01\x00\xfe\xff\xfc\xff\x00\x00\xff\xff\xff\xff\x01\x00\xfe\xff\x00\x00\x00\x00\xfc\xff\xfe\xff\xff\xff\xfe\xff\xfe\xff\xfe\xff\x00\x00\xfd\xff\xfe\xff\x01\x00\xfe\xff\xff\xff\xfe\xff\xfc\xff\x00\x00\xfd\xff\xfc\xff\x00\x00\xfd\xff\xff\xff\xfe\xff\xfd\xff\xfe\xff\xfc\xff\xff\xff\x00\x00\xfe\xff\xff\xff\xfe\xff\x00\x00\x00\x00\x00\x00\x00\x00\xfd\xff\x00\x00\x00\x00\xfb\xff\x00\x00\x00\x00\xfc\xff\x00\x00\x00\x00\xfe\xff\xfd\xff\xff\xff\x00\x00\xfd\xff\x00\x00\x00\x00\xfd\xff\xff\xff\xff\xff\xfd\xff\x00\x00\x00\x00\xfd\xff\x00\x00\x00\x00\xfd\xff\x00\x00\x01\x00\x00\x00\x01\x00\x0

Confirming that the cleaning worked

In [6]:
missing_count = sum(
    text is None or text.strip() == ""
    for text in ds["transcription"]
)

missing_speaker = sum(
    genre == 'Dialog' for genre in ds['type']
)



print(missing_speaker)
print(missing_count)

5000
0


In [7]:
!pip install scipy

We see that we have another problem. The audio is in the form of raw bytes. So we deifine a mapping function to convert to numpy array. ata could also be stereo so we need to change it to mono -> average the two channels

In [8]:
import numpy as np
import io
from scipy.io.wavfile import read

def convert_audio(example):
    current_bytes = example["audio"]["bytes"]
    rate, data= read(io.BytesIO(current_bytes))

    data = data.astype(np.float32) / 32768

    if data.ndim == 2:
        data = np.mean(data, axis = 1)

    return example

In [9]:
ds = ds.map(convert_audio)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [10]:
ds[0]['audio']

{'bytes': b'RIFF\xbe\x05\t\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80\xbb\x00\x00\x00w\x01\x00\x02\x00\x10\x00data\x9a\x05\t\x00\x00\x00\x00\x00\x01\x00\x00\x00\x02\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\xfe\xff\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\xff\xff\x00\x00\xfe\xff\xfe\xff\x01\x00\xfe\xff\xfc\xff\x00\x00\xff\xff\xff\xff\x01\x00\xfe\xff\x00\x00\x00\x00\xfc\xff\xfe\xff\xff\xff\xfe\xff\xfe\xff\xfe\xff\x00\x00\xfd\xff\xfe\xff\x01\x00\xfe\xff\xff\xff\xfe\xff\xfc\xff\x00\x00\xfd\xff\xfc\xff\x00\x00\xfd\xff\xff\xff\xfe\xff\xfd\xff\xfe\xff\xfc\xff\xff\xff\x00\x00\xfe\xff\xff\xff\xfe\xff\x00\x00\x00\x00\x00\x00\x00\x00\xfd\xff\x00\x00\x00\x00\xfb\xff\x00\x00\x00\x00\xfc\xff\x00\x00\x00\x00\xfe\xff\xfd\xff\xff\xff\x00\x00\xfd\xff\x00\x00\x00\x00\xfd\xff\xff\xff\xff\xff\xfd\xff\x00\x00\x00\x00\xfd\xff\x00\x00\x00\x00\xfd\xff\x00\x00\x01\x00\x00\x00\x01\x00\x0

In [11]:
from datasets import Audio

ds = ds.cast_column("audio", Audio(sampling_rate=16000))

In [12]:
ds[0]['audio'].get_all_samples()

AudioSamples:
  data (shape): torch.Size([1, 98543])
  pts_seconds: 0.0
  duration_seconds: 6.1589375
  sample_rate: 16000

In [13]:
from IPython.display import Audio

audio = ds[0]['audio'].get_all_samples()

Audio(audio.data[0], rate = audio.sample_rate)



In [14]:
from transformers import SpeechT5Processor

checkpoint = "microsoft/speecht5_tts"
processor = SpeechT5Processor.from_pretrained(checkpoint)


preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model: reconstructing file:   0%|          |  0.00B /  238kB            

spm_char.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Still not a lot of data especially given that clips/audio are short we can
look at the length of clips longer than a duration and see that


SO MAYBe i get more ata











Any other cleaning stuff i have to do? Maybe identify the ones with no speaker, no trasncirption? - author notes that these are present. Done.


Now just us ethe proessor.


In [15]:
!pip install speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 56.0 MB/s eta 0:00:00


In [16]:
import os
import torch
from speechbrain.pretrained import EncoderClassifier

spk_model_name = "speechbrain/spkrec-xvect-voxceleb"

device = "cuda" if torch.cuda.is_available() else "cpu"

classifier = EncoderClassifier.from_hparams(
    source=spk_model_name,
    run_opts={"device": device},
    savedir=os.path.join("/tmp", spk_model_name),
)

def create_speaker_embedding(waveform):

    embeddings = classifier.encode_batch(waveform)
    embeddings = torch.nn.functional.normalize(embeddings, dim=2)
    embeddings = torch.squeeze(embeddings)

    return embeddings

# embeddings.shape

# # dont wnat mangnitude to be a factor
# torch.Size([1, 1, 512]

# def create_speaker_embedding(example):

#     embeddings = classifier.encode_batch(example)


    # set the paramteers

/tmp/ipykernel_5097/2037695464.py:3: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


hyperparams.yaml:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 16.9MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

...

classifier.ckpt: reconstructing file:   0%|          |  0.00B / 15.9MB            

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


In [17]:
create_speaker_embedding(ds[0]["audio"].get_all_samples().data[0]).shape

torch.Size([512])

In [18]:
def prepare_example(example):

    text = example['transcription']
    audio = example['audio'].get_all_samples()

    # there's a batch dimension on the labels you odn twant
    example = processor(text=text, audio_target=audio.data[0], sampling_rate=audio.sample_rate, return_attention_mask=False)

    example['labels'] = example['labels'][0]

    example['speaker_embeddings'] = create_speaker_embedding(audio.data[0])

    return example

In [19]:
prepare_example(ds[0])

{'input_ids': [4, 51, 8, 9, 31, 6, 4, 20, 8, 13, 13, 22, 23, 4, 24, 15, 5, 7, 12, 5, 4, 6, 5, 15, 15, 4, 18, 5, 4, 20, 11, 7, 6, 4, 22, 8, 16, 4, 28, 9, 8, 20, 44, 4, 32, 11, 10, 12, 4, 5, 27, 5, 9, 6, 26, 26, 26, 4, 30, 4, 11, 7, 27, 5, 4, 7, 15, 13, 5, 7, 14, 22, 4, 12, 5, 7, 13, 17, 11, 5, 14, 4, 19, 8, 13, 4, 7, 9, 12, 20, 5, 13, 12, 4, 19, 8, 13, 4, 7, 4, 15, 8, 9, 21, 4, 6, 10, 18, 5, 26, 2], 'labels': array([[-3.0425184, -3.3300664, -3.6888714, ..., -4.392379 , -4.3512425,
        -4.592181 ],
       [-2.747348 , -3.0614145, -3.1682427, ..., -3.9954262, -3.9184792,
        -4.119288 ],
       [-1.7622545, -1.7358519, -1.7963295, ..., -1.9633925, -1.8851041,
        -1.9930435],
       ...,
       [-2.141781 , -2.2631044, -2.2730918, ..., -3.3227665, -3.4025147,
        -3.4992719],
       [-2.1971283, -2.3485513, -2.2824311, ..., -3.7385995, -3.8302426,
        -3.8940008],
       [-2.4834425, -2.9780617, -2.7674823, ..., -4.1129966, -4.197087 ,
        -4.1479588]], dtype=float

In [20]:
ds = ds.map(prepare_example, remove_columns=ds.column_names)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [21]:
from transformers import SpeechT5ForTextToSpeech

model = SpeechT5ForTextToSpeech.from_pretrained(checkpoint)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  585MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


define collator

In [22]:
from dataclasses import dataclass
from typing import Any
# tells us how to batch our features for our model


@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(self, features):

        # act on them indepdnently share the same key for procesosr

#         input_ids only
# input_values only
# labels only, either log-mel spectrograms or text tokens
# input_ids and log-mel spectrogram labels
# input_values and text labels

        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        input_values =[{"input_values": feature["labels"]} for feature in features]
        speaker_embeddings =[ feature["speaker_embeddings"] for feature in features]

        batch = processor.tokenizer.pad(input_ids, return_tensors='pt')

        feature_extractor = self.processor.feature_extractor

        feature_extractor.feature_size = feature_extractor.num_mel_bins

        labels = feature_extractor.pad(input_values, return_tensors='pt')

        labels = labels['input_values'].masked_fill(labels.attention_mask.unsqueeze(-1).ne(1), -100)

        batch['labels'] = labels

        if model.config.reduction_factor > 1:

            max_length = batch['labels'].shape[1]

            target_length = max_length - max_length % model.config.reduction_factor

            batch["labels"] = batch["labels"][:, :target_length]

        # tensor takes list of lists no dicionary

        batch["speaker_embeddings"] = torch.tensor(speaker_embeddings)

        return batch




Forgot to split oops

In [23]:
splits = ds.train_test_split(test_size=0.2)

In [24]:
# trianing arguments
#

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


data_collator = TTSDataCollatorWithPadding(processor=processor)

arguments =  Seq2SeqTrainingArguments (
    output_dir = "speecht5_finetuned_genshin",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,
    learning_rate = 5e-5,
    max_steps = 2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=2,
    save_steps = 200,
    eval_steps = 200,
    greater_is_better=False,
    logging_steps = 25,
    label_names = ["labels"],
    push_to_hub=True
)



model.safetensors: reconstructing file:   0%|          |  0.00B /  585MB            

model.safetensors: downloading bytes:           |  0.00B            

In [27]:
from huggingface_hub import notebook_login

notebook_login()

In [28]:
collator = TTSDataCollatorWithPadding(processor=processor)

trainer = Seq2SeqTrainer(model=model, eval_dataset=splits["test"], train_dataset=splits["train"], data_collator=collator,
                         args=arguments, processing_class=processor)

In [29]:
trainer.train()

Step,Training Loss,Validation Loss
200,4.960974,0.575730
400,4.638801,0.552574
600,4.565546,0.546885
800,4.461123,0.543455
1000,4.439113,0.538993
1200,4.324223,0.535860
1400,4.281490,0.535045
1600,4.254835,0.536704
1800,4.217615,0.539101


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
200,4.960974,0.575730
400,4.638801,0.552574
600,4.565546,0.546885
800,4.461123,0.543455
1000,4.439113,0.538993
1200,4.324223,0.535860
1400,4.281490,0.535045
1600,4.254835,0.536704
1800,4.217615,0.539101
2000,4.224470,0.537060


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=4.5191708831787105, metrics={'train_runtime': 5065.2778, 'train_samples_per_second': 12.635, 'train_steps_per_second': 0.395, 'total_flos': 7995907768340448.0, 'train_loss': 4.5191708831787105, 'epoch': 16.0})

In [30]:
model = SpeechT5ForTextToSpeech.from_pretrained("chenharry6/speecht5_finetuned_genshin")

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  578MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

In [78]:
from datasets import Audio

text = """You may write me down in history with your bitter, twisted lies. I have a brave soul"""

ds = load_dataset(
    "simon3000/genshin-voice",
    split="train",
    streaming=True,
)

# Cast while the original schema is still available.
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

ds = ds.filter(
    select_rows,
    input_columns=["transcription", "speaker", "type", "language"],
)

# Fetch one matching reference recording.
example = next(iter(ds))

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

In [79]:
from IPython.display import Audio

audio = example['audio'].get_all_samples()

arr = audio.data[0]

Audio(arr, rate = 16000)

In [85]:
embeddings = create_speaker_embedding(arr).unsqueeze(0)
embeddings.shape

torch.Size([1, 512])

In [86]:
inputs = processor(text = text, return_tensors="pt")

In [87]:
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
vocoder = vocoder.to(device).eval()
inputs = inputs.to(device)
embeddings = embeddings.to(device=device, dtype=torch.float32)

[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [88]:
from transformers import SpeechT5HifiGan

speech = model.generate_speech(inputs["input_ids"], speaker_embeddings=embeddings, vocoder = vocoder)

In [89]:
speech = speech.cpu().numpy()

In [91]:
Audio(speech, rate=16000)